In [29]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

import torch

In [2]:
with open("../data/bhagavad_gita_sft_no_sanskrit.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} conversations")

Loaded 620 conversations


In [3]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [4]:
def load_tokenizer(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Tokenizer file not found: {path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Restore ordered BPE merge rules.
    merges = {
        (first_id, second_id): new_id
        for first_id, second_id, new_id in data["merges"]
    }

    vocab = build_vocab(merges)

    return merges, vocab, data["vocab_size"]

In [5]:
tokenizer_path = Path("tokenizer/tokenizer/tokenizer.json")
if not tokenizer_path.exists():
    tokenizer_path = Path("..") / tokenizer_path

merges, vocab, vocab_size = load_tokenizer(tokenizer_path)

print("Tokenizer loaded")
print("Vocabulary size:", vocab_size)
print("Number of merges:", len(merges))

Tokenizer loaded
Vocabulary size: 1000
Number of merges: 744


In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def build_fast_tokenizer(merges, vocab):
    byte_values = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    unicode_values = byte_values.copy()
    extra_index = 0

    for byte_value in range(256):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(256 + extra_index)
            extra_index += 1

    byte_encoder = {
        byte_value: chr(unicode_value)
        for byte_value, unicode_value
        in zip(byte_values, unicode_values)
    }

    def bytes_to_token_string(byte_sequence):
        return "".join(
            byte_encoder[byte_value]
            for byte_value in byte_sequence
        )

    fast_vocab = {
        bytes_to_token_string(byte_sequence): token_id
        for token_id, byte_sequence in vocab.items()
    }

    fast_merges = [
        (
            bytes_to_token_string(vocab[first_id]),
            bytes_to_token_string(vocab[second_id]),
        )
        for first_id, second_id in merges
    ]

    tokenizer = Tokenizer(
        BPE(
            vocab=fast_vocab,
            merges=fast_merges,
        )
    )

    tokenizer.pre_tokenizer = ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    )

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


fast_tokenizer = build_fast_tokenizer(
    merges=merges,
    vocab=vocab,
)

print("Fast tokenizer created")
print("Vocabulary size:", fast_tokenizer.get_vocab_size())

Fast tokenizer created
Vocabulary size: 1000


In [7]:
def format_conversation(conversation):
    text = ""

    for message in conversation["messages"]:
        role = message["role"]
        content = message["content"]

        text += f"{role}: {content}\n"

    return text

In [8]:
all_input_ids = []
all_text = ""
for conversation in data:
    text = format_conversation(conversation)
    all_text += text + "<newLINE>"+  "\n" 

    encoding = fast_tokenizer.encode(text)

    all_input_ids.append(encoding.ids)

In [9]:
print("Conversations:", len(data))
print("Encoded conversations:", len(all_input_ids))

Conversations: 620
Encoded conversations: 620


In [10]:
lengths = [len(ids) for ids in all_input_ids]

print("Total conversations:", len(all_input_ids))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

Total conversations: 620
Minimum tokens: 164
Maximum tokens: 3208
Average tokens: 1285.116129032258


In [11]:
import numpy as np

print("P50:", np.percentile(lengths, 50))
print("P75:", np.percentile(lengths, 75))
print("P90:", np.percentile(lengths, 90))
print("P95:", np.percentile(lengths, 95))
print("P99:", np.percentile(lengths, 99))

P50: 1247.0
P75: 1526.25
P90: 1823.3000000000002
P95: 2068.1
P99: 2479.43


In [12]:
import re

CONTEXT_LENGTH = 512
MAX_USER_TOKENS = 350


def token_length(text):
    return len(fast_tokenizer.encode(text).ids)


def format_pair(user_text, assistant_text):
    return (
        f"user: {user_text}\n"
        f"assistant: {assistant_text}"
    )


def shorten_user_prompt(user_text, max_tokens=MAX_USER_TOKENS):
    user_ids = fast_tokenizer.encode(user_text).ids

    if len(user_ids) <= max_tokens:
        return user_text, False

    user_ids = user_ids[-max_tokens:]

    return fast_tokenizer.decode(user_ids).strip(), True


def split_into_sentences(text):
    text = text.strip()

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    parts = re.split(
        r'\n\s*\n|(?<=[.!?])\s+',
        text
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


def hard_split_text(text, prefix, context_length=512):
    words = text.split()

    chunks = []
    current_words = []

    for word in words:
        candidate_words = current_words + [word]
        candidate = " ".join(candidate_words)

        if token_length(prefix + candidate) <= context_length:
            current_words.append(word)

        else:
            if current_words:
                chunks.append(" ".join(current_words))

            current_words = [word]

            if token_length(prefix + word) > context_length:
                current_words = []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def split_assistant_response(
    user_text,
    assistant_text,
    context_length=512
):
    prefix = (
        f"user: {user_text}\n"
        f"assistant: "
    )

    if token_length(prefix) >= context_length:
        return []

    sentences = split_into_sentences(assistant_text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if current_chunk:
            candidate = current_chunk + " " + sentence
        else:
            candidate = sentence

        if token_length(prefix + candidate) <= context_length:
            current_chunk = candidate

        else:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            if token_length(prefix + sentence) <= context_length:
                current_chunk = sentence

            else:
                smaller_chunks = hard_split_text(
                    sentence,
                    prefix,
                    context_length
                )

                chunks.extend(smaller_chunks)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


def create_training_samples(
    cleaned_data,
    context_length=512
):
    training_samples = []

    stats = {
        "original_conversations": len(cleaned_data),
        "pairs_found": 0,
        "pairs_that_fit": 0,
        "pairs_split": 0,
        "user_prompts_shortened": 0,
        "samples_created": 0,
        "skipped": 0
    }

    for conversation in cleaned_data:
        messages = conversation.get("messages", [])

        i = 0

        while i < len(messages) - 1:
            current = messages[i]
            next_message = messages[i + 1]

            if (
                current.get("role") == "user"
                and next_message.get("role") == "assistant"
            ):
                user_text = current["content"].strip()
                assistant_text = next_message["content"].strip()

                stats["pairs_found"] += 1

                if not user_text or not assistant_text:
                    stats["skipped"] += 1
                    i += 2
                    continue

                user_text, was_shortened = shorten_user_prompt(
                    user_text
                )

                if was_shortened:
                    stats["user_prompts_shortened"] += 1

                full_text = format_pair(
                    user_text,
                    assistant_text
                )

                encoding = fast_tokenizer.encode(full_text)

                if len(encoding.ids) <= context_length:
                    training_samples.append({
                        "text": full_text,
                        "input_ids": encoding.ids,
                        "user": user_text,
                        "assistant": assistant_text,
                        "was_split": False,
                        "user_was_shortened": was_shortened
                    })

                    stats["pairs_that_fit"] += 1

                else:
                    assistant_chunks = split_assistant_response(
                        user_text,
                        assistant_text,
                        context_length
                    )

                    if not assistant_chunks:
                        stats["skipped"] += 1
                        i += 2
                        continue

                    stats["pairs_split"] += 1

                    for chunk in assistant_chunks:
                        text = format_pair(
                            user_text,
                            chunk
                        )

                        encoding = fast_tokenizer.encode(text)

                        if len(encoding.ids) <= context_length:
                            training_samples.append({
                                "text": text,
                                "input_ids": encoding.ids,
                                "user": user_text,
                                "assistant": chunk,
                                "was_split": True,
                                "user_was_shortened": was_shortened
                            })

                i += 2

            else:
                i += 1

    stats["samples_created"] = len(training_samples)

    return training_samples, stats

In [13]:
training_samples, stats = create_training_samples(
    data,
    context_length=512
)

In [14]:
stats

{'original_conversations': 620,
 'pairs_found': 1114,
 'pairs_that_fit': 265,
 'pairs_split': 849,
 'user_prompts_shortened': 5,
 'samples_created': 2745,
 'skipped': 0}

In [15]:
lengths = [
    len(sample["input_ids"])
    for sample in training_samples
]

print("\nTotal training samples:", len(training_samples))
print("Minimum length:", min(lengths))
print("Maximum length:", max(lengths))
print("Average length:", sum(lengths) / len(lengths))

assert max(lengths) <= 512

print("\nAll samples are <= 512 tokens.")


Total training samples: 2745
Minimum length: 52
Maximum length: 512
Average length: 420.28123861566485

All samples are <= 512 tokens.


In [16]:
split_examples = [
    sample
    for sample in training_samples
    if sample["was_split"]
]

print("Split samples:", len(split_examples))

Split samples: 2480


In [17]:
for sample in split_examples[:3]:

    print("=" * 100)

    print("TOKENS:", len(sample["input_ids"]))

    print("\nUSER:")
    print(sample["user"])

    print("\nASSISTANT CHUNK:")
    print(sample["assistant"])

    print()

TOKENS: 507

USER:
The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?

ASSISTANT CHUNK:
Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34: (Tad viddhi pranipatena pariprasnena sevaya. Upadeksyanti te jnanam jnaninas tattva-darsinah.)This verse translates to: 'Learn the truth by approaching a spiritual master. Inquire from him submissive

In [18]:
long_users = []

for conversation in data:

    for message in conversation["messages"]:

        if message["role"] == "user":

            text = f"user: {message['content']}\nassistant: "

            length = token_length(text)

            if length > 350:
                long_users.append(length)


print("User prompts > 350 tokens:", len(long_users))

if long_users:
    print("Longest user prefix:", max(long_users))

User prompts > 350 tokens: 7
Longest user prefix: 721


In [21]:
print(training_samples[:2])

[{'text': "user: The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?\nassistant: Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34: (Tad viddhi pranipatena pariprasnena sevaya. Upadeksyanti te jnanam jnaninas tattva-darsinah.)This verse translates to: 'Learn the truth by approaching a spiritual master. Inquire from him submissively and r

In [22]:
def add_labels(training_samples):
    processed_samples = []

    for sample in training_samples:
        user_text = sample["user"]
        assistant_text = sample["assistant"]

        prefix = (
            f"user: {user_text}\n"
            f"assistant: "
        )

        full_text = prefix + assistant_text

        full_encoding = fast_tokenizer.encode(full_text)
        prefix_encoding = fast_tokenizer.encode(prefix)

        input_ids = full_encoding.ids.copy()

        labels = input_ids.copy()

        prefix_length = len(prefix_encoding.ids)

        for i in range(prefix_length):
            labels[i] = -100

        processed_samples.append({
            "input_ids": input_ids,
            "labels": labels
        })

    return processed_samples

In [23]:
processed_samples = add_labels(training_samples)

In [26]:
print("Input IDs:")
print(processed_samples[0]["input_ids"])

print("\nLabels:")
print(processed_samples[0]["labels"])

Input IDs:
[451, 264, 58, 32, 375, 568, 115, 294, 902, 104, 552, 320, 380, 673, 366, 273, 589, 112, 405, 822, 109, 115, 470, 335, 116, 355, 426, 340, 105, 603, 423, 366, 772, 108, 912, 279, 839, 989, 512, 324, 39, 115, 276, 568, 105, 39, 32, 284, 32, 103, 117, 297, 101, 44, 769, 495, 809, 455, 105, 304, 416, 637, 280, 112, 673, 117, 262, 275, 336, 101, 107, 859, 514, 642, 445, 289, 382, 119, 775, 501, 115, 259, 959, 100, 305, 488, 422, 432, 514, 542, 117, 525, 298, 515, 103, 110, 105, 122, 256, 115, 117, 330, 289, 103, 117, 297, 388, 367, 112, 827, 582, 269, 924, 307, 293, 117, 97, 327, 348, 986, 115, 496, 594, 301, 268, 637, 335, 435, 115, 344, 110, 634, 440, 287, 362, 335, 283, 117, 98, 109, 379, 316, 275, 483, 321, 115, 100, 479, 560, 273, 589, 106, 816, 364, 263, 335, 283, 480, 102, 45, 296, 112, 405, 749, 109, 476, 274, 102, 764, 316, 878, 298, 99, 974, 279, 44, 259, 262, 410, 635, 261, 389, 66, 281, 478, 118, 97, 257, 71, 293, 97, 63, 10, 97, 684, 105, 304, 287, 116, 58, 32, 73, 

In [28]:
CONTEXT_LENGTH = 512
PAD_TOKEN_ID = 1000


def pad_samples(processed_samples, context_length=512):
    padded_samples = []

    for sample in processed_samples:
        input_ids = sample["input_ids"]
        labels = sample["labels"]

        length = len(input_ids)

        padding_length = context_length - length

        padded_input_ids = (
            input_ids
            + [PAD_TOKEN_ID] * padding_length
        )

        attention_mask = (
            [1] * length
            + [0] * padding_length
        )

        padded_labels = (
            labels
            + [-100] * padding_length
        )

        padded_samples.append({
            "input_ids": torch.tensor(
                padded_input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                padded_labels,
                dtype=torch.long
            )
        })

    return padded_samples

In [30]:
padded_samples = pad_samples(
    processed_samples,
    context_length=512
)

In [31]:
print(padded_samples[0]["input_ids"].shape)
print(padded_samples[0]["attention_mask"].shape)
print(padded_samples[0]["labels"].shape)

torch.Size([512])
torch.Size([512])
torch.Size([512])


In [32]:
sample = padded_samples[0]

print("Input IDs:")
print(sample["input_ids"])

print("\nAttention mask:")
print(sample["attention_mask"])

print("\nLabels:")
print(sample["labels"])

Input IDs:
tensor([ 451,  264,   58,   32,  375,  568,  115,  294,  902,  104,  552,  320,
         380,  673,  366,  273,  589,  112,  405,  822,  109,  115,  470,  335,
         116,  355,  426,  340,  105,  603,  423,  366,  772,  108,  912,  279,
         839,  989,  512,  324,   39,  115,  276,  568,  105,   39,   32,  284,
          32,  103,  117,  297,  101,   44,  769,  495,  809,  455,  105,  304,
         416,  637,  280,  112,  673,  117,  262,  275,  336,  101,  107,  859,
         514,  642,  445,  289,  382,  119,  775,  501,  115,  259,  959,  100,
         305,  488,  422,  432,  514,  542,  117,  525,  298,  515,  103,  110,
         105,  122,  256,  115,  117,  330,  289,  103,  117,  297,  388,  367,
         112,  827,  582,  269,  924,  307,  293,  117,   97,  327,  348,  986,
         115,  496,  594,  301,  268,  637,  335,  435,  115,  344,  110,  634,
         440,  287,  362,  335,  283,  117,   98,  109,  379,  316,  275,  483,
         321,  115,  100,  47

In [33]:
for sample in padded_samples:
    assert len(sample["input_ids"]) == 512
    assert len(sample["attention_mask"]) == 512
    assert len(sample["labels"]) == 512

print("All samples are padded to 512.")

All samples are padded to 512.


In [34]:
from sklearn.model_selection import train_test_split

train_samples, val_samples = train_test_split(
    padded_samples,
    test_size=0.05,
    random_state=42
)

print("Train samples:", len(train_samples))
print("Validation samples:", len(val_samples))

Train samples: 2607
Validation samples: 138


In [35]:
from torch.utils.data import Dataset


class SFTDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [36]:
train_dataset = SFTDataset(train_samples)
val_dataset = SFTDataset(val_samples)

In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [39]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([32, 512])
torch.Size([32, 512])
torch.Size([32, 512])
